In [3]:
import csv
from pathlib import Path

csv_path = Path("pump_recipe.csv")

# Exactly 12 headers (ports 1-12). One must be MAIN and one must be OUT.
headers = [
    "MAIN",
    "H2O2",
    "KCl",
    "H2O",
    "EtOH",
    "Acetone",
    "MeOH",
    "Buffer",
    "Catalyst",
    "Ligand",
    "NaOH",
    "OUT",
]

# One recipe row: empty cells are ignored.
# MAIN/OUT columns identify destination lines and are typically left blank.
recipe_row = [
    "",  # Port 1 (NaOH)
    "",
    10,  # Port 3 (KCl)
    "",
    "",
    "",  # Port 6 (Acetone)
    "",
    "",
    "",
    "",
    "",  
    "",  
]

with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(headers)
    writer.writerow(recipe_row)

print(f"Created recipe CSV: {csv_path.resolve()}")

Created recipe CSV: /mnt/storage/Projects/ACSDL1/AC-OTFlex-monorepo/devices/syringe-pump/src/pump_recipe.csv


In [4]:
import importlib.util
from pathlib import Path

# Resolve syringe-pump.py from common working directories
candidate_paths = [
    Path.cwd() / "syringe-pump.py",
    Path.cwd() / "devices/syringe-pump/src/syringe-pump.py",
    Path("devices/syringe-pump/src/syringe-pump.py"),
]
module_path = next((p.resolve() for p in candidate_paths if p.exists()), None)

if module_path is None:
    raise FileNotFoundError("Could not locate syringe-pump.py")

spec = importlib.util.spec_from_file_location("syringe_pump_module", module_path)
syringe_pump_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(syringe_pump_module)

speed = 1.0

result = syringe_pump_module.run_csv_protocol(
    csv_path=str(csv_path),
    speed=speed,
)

main_port = result["main_port"]
out_port = result["out_port"]
operations = result["operations"]

print(f"Detected MAIN port: {main_port}")
print(f"Detected OUT port: {out_port}")
print(f"Executed {len(operations)} transfer(s) to MAIN")

for op in operations:
    print(
        f"Row {op['row']} | {op['channel_label']} | "
        f"draw_port={op['draw_valve_port']} | volume_mL={op['volume_ml']}"
    )

Detected MAIN port: 1
Detected OUT port: 12
Executed 1 transfer(s) to MAIN
Row 2 | KCl | draw_port=3 | volume_mL=10.0
